<a href="https://colab.research.google.com/github/Condemor-bit/SEFAC_IA/blob/main/chatbot_openai_gpt_oss_20b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Preparar versión de torch y transformers para que funcionen con MXFP4 quantization gpt-oss-20b
#!pip install -q --upgrade torch
!pip uninstall torch -y
!pip install  torch==2.9.1 git+https://github.com/huggingface/transformers.git@fe5ca9d triton==3.5.1 kernels==0.11.0
!pip uninstall -q torchvision torchaudio -y

In [ ]:
MEMORY = False
RAZONAMIENTO = True
ALTURA = 600

In [ ]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
import torch
from threading import Thread

# Modelo y tokenizador
model_name = "openai/gpt-oss-20b"

def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype="auto"
    )
    return model, tokenizer

# Cargar el modelo y el tokenizador
model, tokenizer = load_model(model_name)



def generate_response(user_message, max_tokens, temperature, top_k, top_p, repetition_penalty, history_state, reasoning_level_dropdown):
    if not user_message.strip():
        return history_state, history_state


    global model, tokenizer

    system_message = (
    f"You are an advanced reasoning assistant.\n"
    f"Reasoning: {reasoning_level_dropdown}\n")

    # Construir mensajes en formato correcto
    messages = [{"role": "system", "content": system_message}]


    ###################
    ###################
    ###################
    #PARA  TENER MEMORIA DE LA CONVERSACIÓN
    if MEMORY==True:
      messages.extend(history_state)
    ###################
    ###################
    ###################

    # Añadir mensaje actual del usuario
    messages.append({"role": "user", "content": user_message})

    # Convertir prompt a tensores
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True
    )
    #pasar los tensores al device = GPU/CPU
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Crear streamer para obtener tokens a medida que se generan
    streamer = TextIteratorStreamer(tokenizer, skip_special_tokens=True)

    #parametros de generación de texto (se obtienen de los parametros que introduce el usuario en gradio)
    generation_kwargs = {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "max_new_tokens": int(max_tokens),
        "do_sample": True,
        "temperature": float(temperature),
        "top_k": int(top_k),
        "top_p": float(top_p),
        "repetition_penalty": float(repetition_penalty),
        "streamer": streamer
    }

    # Función para generar el resultado sin calculo de gradientes (no es necesario si no se entrena). reduce el uso de memoria
    def generate_in_thread(**kwargs):
        with torch.no_grad():
            model.generate(**kwargs)

    # Ejecutar generación en paralelo.
    thread = Thread(target=generate_in_thread, kwargs=generation_kwargs)
    #thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # Stream de tokens
    assistant_response = ""
    new_history = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": ""}
        ]

    #### VER RAZONAMIENTO
    if RAZONAMIENTO==True:
      for token in streamer:
        assistant_response += token
        new_history[-1]["content"] = assistant_response.strip()
        yield new_history, new_history

    ###########################
    ###########################
    #### OCULTAR RAZONAMIENTO
    else:
      for token in streamer:
          assistant_response += token

          display_text = ""

          if "assistantfinal" in assistant_response:
              display_text = assistant_response.split("assistantfinal")[-1].strip()

          elif "assistantanalysis" in assistant_response:
              # Si está en fase de análisis, podemos mostrar un mensaje de espera
              display_text = "🤔 Pensando..."

          else:
              # Si es el inicio y hay basura del sistema, intentamos no mostrar nada aún
              # o mostramos lo que hay si no detectamos estructura rara.
              if "systemYou" in assistant_response:
                  display_text = "..."
              else:
                  display_text = assistant_response.strip()

          # Actualizamos el historial visual con el texto limpio
          new_history[-1]["content"] = display_text

          # Devolvemos new_history tanto para el chat visual como para el estado
          yield new_history, new_history
    ###########################

    # Para que en la siguiente vuelta no se vuelva a meter la basura del sistema al prompt.
    if "assistantfinal" in assistant_response:
        final_clean_response = assistant_response.split("assistantfinal")[-1].strip()
        new_history[-1]["content"] = final_clean_response


    thread.join()  # Asegura que el hilo termine

    # Liberar explícitamente tensores y caché
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


with gr.Blocks(theme=gr.themes.Ocean(), fill_width=True, fill_height=True) as demo:
    gr.Markdown(
        """
        # Bienvenido a tu primer ChatBot con gpt-oss-20b de openai.
        """
    )

    history_state = gr.State([])

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Settings")
            max_tokens_slider = gr.Slider(
                minimum=5,
                maximum=32768,
                step=1024,
                value=4096,
                label="Max Tokens"
            )
            with gr.Accordion("Advanced Settings", open=False):
                temperature_slider = gr.Slider(
                    minimum=0.1,
                    maximum=1.5,
                    value=0.8,
                    label="Temperature"
                )
                top_k_slider = gr.Slider(
                    minimum=1,
                    maximum=100,
                    step=1,
                    value=50,
                    label="Top-k"
                )
                top_p_slider = gr.Slider(
                    minimum=0.1,
                    maximum=1.0,
                    value=0.95,
                    label="Top-p"
                )
                repetition_penalty_slider = gr.Slider(
                    minimum=1.0,
                    maximum=1.5,
                    value=1.05,
                    label="Repetition Penalty"
                )
                reasoning_level_dropdown = gr.Dropdown(
                    choices=["low", "medium", "high"],
                    value="medium",
                    label="Reasoning Level"
                )


        with gr.Column(scale=8):
            chatbot = gr.Chatbot(label="Chat",
                                 type="messages",
                                 height=ALTURA, ### tamaño del chat
                                 )
            with gr.Row():
                user_input = gr.Textbox(
                    label="Your message",
                    placeholder="Type your message here...",
                    scale=5
                )
                submit_button = gr.Button("Send", variant="primary", scale=1)
                clear_button = gr.Button("Clear", scale=1)

    submit_button.click(
        fn=generate_response,
        inputs=[user_input, max_tokens_slider, temperature_slider, top_k_slider, top_p_slider, repetition_penalty_slider, history_state, reasoning_level_dropdown],
        outputs=[chatbot, history_state]
    ).then(
        fn=lambda: gr.update(value=""),
        inputs=None,
        outputs=user_input
    )

    clear_button.click(
        fn=lambda: ([], []),
        inputs=None,
        outputs=[chatbot, history_state]
    )

creatividad:
temperature = 1.1
top_k = 60
top_p = 0.97
repetition_penalty = 1.1

menos creatividad:
temperature = 0.4
top_k = 30
top_p = 0.9
repetition_penalty = 1.05

equilibrado:
temperature = 0.7
top_k = 50
top_p = 0.95
repetition_penalty = 1.1

In [ ]:
# Lanzar con logs visibles
demo.launch(
    share=True, # Nos permite conectarnos en remoto a la interfaz.
    debug=True,  # Activa modo debug
    show_error=True  # Muestra errores en la interfaz
)